# Heterogeneous Graph Visualization

Visualize the bipartite graph structure that separates spacetime from field content.

In [ ]:
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
    !pip install -q torch-geometric omegaconf
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
else:
    sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from qft_graph.config import LatticeConfig
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.fields.scalar import ScalarField
from qft_graph.graphs.builder import HeteroGraphBuilder
from qft_graph.graphs.edge_types import ADJACENT, inhabits_edge

plt.style.use('dark_background')
%matplotlib inline

In [ ]:
# Use small lattice for visibility
lattice = HypercubicLattice(LatticeConfig(dimensions=(4, 4)))
scalar = ScalarField()
builder = HeteroGraphBuilder(lattice, [scalar])

torch.manual_seed(42)
phi = scalar.initialize(lattice.num_sites(), mode='hot')
data = builder.build({'scalar': phi})

print('Node types:', data.node_types)
print('Edge types:', data.edge_types)
print(f'Spacetime nodes: {data["spacetime"].num_nodes}')
print(f'Scalar nodes: {data["scalar"].num_nodes}')
print(f'Adjacency edges: {data[ADJACENT].edge_index.shape[1]}')
print(f'Inhabits edges: {data[inhabits_edge("scalar")].edge_index.shape[1]}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

coords = data['spacetime'].x[:, :2].numpy()  # x, y coordinates
phi_vals = data['scalar'].x[:, 0].numpy()

# Offset field nodes below spacetime nodes
field_offset = -1.5
field_coords = coords.copy()
field_coords[:, 1] += field_offset

# Draw adjacency edges (spacetime-spacetime)
adj = data[ADJACENT].edge_index.numpy()
for i in range(adj.shape[1]):
    src, dst = adj[0, i], adj[1, i]
    # Skip wrap-around edges for cleaner visualization
    if np.linalg.norm(coords[src] - coords[dst]) < 1.5:
        ax.plot([coords[src, 0], coords[dst, 0]],
                [coords[src, 1], coords[dst, 1]],
                color='#4488ff', alpha=0.3, linewidth=1)

# Draw inhabits edges (field-spacetime)
for i in range(len(coords)):
    ax.plot([field_coords[i, 0], coords[i, 0]],
            [field_coords[i, 1], coords[i, 1]],
            color='#ff8844', alpha=0.4, linewidth=0.8, linestyle='--')

# Draw spacetime nodes
ax.scatter(coords[:, 0], coords[:, 1], s=200, c='#4488ff',
           edgecolors='white', linewidth=1.5, zorder=5, label='Spacetime nodes')

# Draw field nodes (color by phi value)
sc = ax.scatter(field_coords[:, 0], field_coords[:, 1], s=200,
                c=phi_vals, cmap='RdBu_r', edgecolors='white',
                linewidth=1.5, zorder=5, label='Scalar field nodes')
plt.colorbar(sc, ax=ax, label=r'$\phi$ value', shrink=0.6)

# Labels
for i in range(len(coords)):
    ax.annotate(f'ST{i}', coords[i], fontsize=6, ha='center', va='center', color='white', fontweight='bold')
    ax.annotate(f'{phi_vals[i]:.1f}', field_coords[i], fontsize=6, ha='center', va='center', color='white', fontweight='bold')

ax.set_xlim(-0.5, 3.5)
ax.set_title('Heterogeneous Bipartite Graph (4x4 lattice)', fontsize=14)

# Legend
handles = [
    mpatches.Patch(color='#4488ff', label='Spacetime nodes + adjacency'),
    mpatches.Patch(color='#ff8844', label='Inhabits edges (bipartite)'),
    mpatches.Patch(color='#cc4444', label='Scalar field nodes'),
]
ax.legend(handles=handles, loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()